# Newton (XPBD) vs. FEniCSx (FEM) — run everything on Kaggle

Kaggle gives a **better free GPU** than Colab (P100 16 GB, or 2× T4) and a real
**API** (`kaggle kernels push/output`) so the whole pipeline can be run headlessly.
This notebook installs both stacks, pulls the repo and runs every stage; the
quantitative evaluation lives in the `10_`/`20_`/`30_`/`40_` analysis notebooks.

**Before running:** open the right-hand panel and set
* **Accelerator → GPU** (P100 or T4 x2), and
* **Internet → On** (needed for pip + git clone).

To run via the API instead, see `kaggle/kernel-metadata.json` and the README
(`kaggle kernels push -p .`).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Newton + Warp

NVIDIA Warp (`warp-lang`) is Newton's core dependency and is pulled in
automatically. Internet must be ON.

In [ ]:
!pip install -q "newton[examples]" matplotlib
import warp as wp; wp.init(); print('warp', wp.config.version, '| device:', wp.get_device())

## 3. Install FEniCSx (dolfinx) via fem-on-colab

The `fem-on-colab` real-mode build also works on Kaggle. If the stack clashes
with Warp in one session, run the FEM modules in a **separate** Kaggle notebook —
the two sides only exchange `data/*.npz`.

In [ ]:
import os
# fem-on-colab dolfinx build (works on Colab AND Kaggle). NOTE: the URL is the
# 'release' build -- the old fenicsx-install-real.sh path is gone (returns 404).
URL = "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"
!wget -q "{URL}" -O /tmp/fenicsx.sh
assert os.path.getsize("/tmp/fenicsx.sh") > 0, "installer download failed (0 bytes) -- check URL / Internet"
!bash /tmp/fenicsx.sh
import dolfinx, mpi4py, petsc4py; print("dolfinx", dolfinx.__version__)

## 4. Get the project files & install (editable)

Clone the repo (Internet ON) and `pip install -e .` so `common`, `newton_run`,
`fenics_run`, `compare` import from anywhere.

In [ ]:
import os
DST = '/kaggle/working/Newton'
REPO = 'https://github.com/AustrianTradingMachine/Newton.git'   # public repo
# Clone the public repo (Internet must be ON). Re-runs reuse the existing checkout.
if not os.path.isdir(DST):
    !git clone --depth 1 {REPO} {DST}
%cd /kaggle/working/Newton
!pip install -q -e .
print('cwd:', os.getcwd()); print(os.listdir('.'))

## 5. Stage A — hanging block, BOTH Newton solvers + FEM + analytic

XPBD (writes the shared mesh), the explicit/SemiImplicit solver, and FEM tet/hex.

In [ ]:
!python -m newton_run.run_stage_a                       # XPBD (shared mesh)
!python -m newton_run.run_stage_a --solver semi_implicit  # explicit solver
!python -m fenics_run.run_stage_a --element tet
!python -m fenics_run.run_stage_a --element hex
!python -m compare.metrics
from IPython.display import Image, display
for f in ['stage_a_profile.png', 'stage_a_settling.png', 'stage_a_error_hist.png']:
    display(Image('figures/' + f))

## 6. Stage B — rigid-sphere contact (FEM penalty/AL + Newton XPBD)

In [ ]:
!python -m fenics_run.run_stage_b
!python -m newton_run.run_stage_b
!python -m compare.stage_b
from IPython.display import Image, display
for f in ['stage_b_force.png', 'stage_b_penetration.png', 'stage_b_compare_profile.png']:
    display(Image('figures/' + f))

## 7. (4) Convergence study — Newton iters/substeps + FEM h-refinement

Discretisation error vs solver error; analysis in `30_convergence_analysis.ipynb`.

In [ ]:
!python -m newton_run.convergence_stage_a   # CUDA
!python -m fenics_run.convergence_stage_a   # needs dolfinx
!python -m compare.convergence
from IPython.display import Image, display
import os
for f in ['convergence_newton.png', 'convergence_fem.png']:
    if os.path.exists('figures/' + f): display(Image('figures/' + f))

## 8. Friction — sliding block, Coulomb

FEM friction force + dissipated work vs the analytic `mu*W`; Newton slip
(`soft_contact_mu`, no calibrated force). Analysis in `40_friction_analysis.ipynb`.

In [ ]:
!python -m fenics_run.run_friction   # needs dolfinx
!python -m newton_run.run_friction   # CUDA
!python -m compare.friction
from IPython.display import Image, display
import os
for f in ['friction_force.png', 'friction_work.png', 'friction_slip.png']:
    if os.path.exists('figures/' + f): display(Image('figures/' + f))

## 9. (Optional) differentiable fit, dynamic drop, stress-strain

In [ ]:
!python -m newton_run.diffsim_stage_a            # differentiable stiffness fit
!python -m newton_run.example_rigid_soft_contact # literal XPBD drop
!python -m fenics_run.run_drop                   # FEM Newmark drop
!python -m compare.drop
!python -m fenics_run.run_stress_strain
!python -m newton_run.run_stress_strain
!python -m compare.stress_strain

## 10. Evaluation

All `data/*.npz` are now written. Open the analysis notebooks for the full
quantitative evaluation:

* **`10_stage_a_analysis.ipynb`** — deflection, profile, energies, Jacobian,
  dissipation, equilibrium residual, **FEM-vs-analytic validity**, both Newton solvers.
* **`20_stage_b_analysis.ipynb`** — contact force, dimple, contact energy, penetration.
* **`30_convergence_analysis.ipynb`** — convergence / discretisation vs solver error.
* **`40_friction_analysis.ipynb`** — friction force, Coulomb plateau, dissipated work.

With the Kaggle API you can also fetch the outputs headlessly:
`kaggle kernels output <user>/newton-vs-fem -p ./out`.

## 11. (Optional) Push results back with kagglehub

Ship the produced `data/` + `figures/` back out as a versioned Kaggle Dataset, so
you can pull them locally without the `kaggle kernels output` CLI. Needs your
Kaggle credentials in the kernel (Add-ons → Secrets, or `kagglehub.login()`).

In [ ]:
# !pip install -q kagglehub
# Push data/ + figures/ as <user>/newton-vs-fem-out (set your username):
# !python kaggle/kagglehub_sync.py upload-out --prefix <user>
# Then LOCALLY pull them:
#   python kaggle/kagglehub_sync.py download-out --prefix <user> -o ./out
print('kagglehub transport helper: kaggle/kagglehub_sync.py')